# 1: Introduction to Causality

This introduction is for data and machine learning professionals looking to add causality to their toolkit, as well as those sceptical of its practical use.

The chapter explains why causality matters through motivating examples, provides a brief history of the field, and explores current industry applications.
Crucially, it describes the mental shift required when moving from a typical machine learning workflow to a causal one.

##### Background Knowledge
We assume familiarity with standard statistics, specifically correlation and logistic regression.
If you need a refresher, we recommend Chapters 3 and 4 of **Introduction to Statistical Learning**<a name="intro_stat_learn"></a>[<sup>[1]</sup>](#intro_stat_learn), also available at https://www.statlearning.com/.

**Ch.3: Linear Regression**
1. Simple Linear Regression
2. Multiple Linear Regression
3. Other Considerations in the Regression Model
4. The Marketing Plan
5. Comparison of Linear Regression with $k$-Nearest Neighbours
6. Lab: Linear Regression
7. Exercises

**Ch.4: Classification**
1. An Overview of Classification
2. Why Not Linear Regression?
3. Logistic Regression
4. Generative Models for Classification
5. A Comparison of Classification Methods
6. Generalised Linear Models
7. Lab: Logistic Regression. LDA, QDA, and KNN
8. Exercises



<a name="intro_stat_learn"></a>1. [^](#intro_stat_learn) _Gareth James_, _Daniela Witten_, _Trevor Hastie_, and _Robert Tibshirani_. 2013. **An Introduction to Statistical Learning.** Springer. https://doi.org/10.1007/978-1-4614-7138-7.


## 1.1: Why Should I Use Causality?

Data availability and the questions we ask of it have grown exponentially over the last 20 years.
Most of these questions are fundamentally causal:

* **Why** did metric $X$ change this month?
* **Which** customers should we target with $Y$ campaign?
* **What** would happen if we instituted $Z$ policy change?

Modern companies frequently use data to quantify causal relationships.
While standard correlational analysis explains _what_ happened, it often fails to explain _why_ or _how_.
Relying on correlation alone can be misleading due to two main challenges: difficulty in interpreting associations and the risk of effects appearing to reverse.
Causal inference provides the tools to solve both.

<center>
<img src="images/causal_vs_correl.jpeg" alt="Confusing correlation with causation can be misleading." width="400"/>
</center>

> The "reversed effects" refer to Simpson's Paradox, where a trend appearing in different groups of data disappears or reverses when the groups are combined.
> Causal inference provides the framework to determine which level of the data reveals the true relationship.

Interest in causal methods is growing beyond traditional statistics, as shown by the exponential rise in causal research at major machine learning conferences like NeurIPS.
 This momentum reflects a broader shift toward using causality to solve complex problems that standard ML cannot.


### 1.1.1: Challenge #1: _Correlation Does Not Imply Causation_

While _"correlation does not imply causation"_ is a cliché, it remains a fundamental truth: strong associations in data do not automatically mean one variable causes the other.
A classic example is the correlation between ice cream sales and sunburn or shark attacks; the data might suggest a link, but eating ice cream obviously doesn't cause sunburn or shark attacks.

<center>
<img src="images/common_cause.png" alt="Correlation does not imply causation." width="400"/>
</center>

Some associations aren't linked by a common cause or any real-world logic at all.
These are known as **spurious correlations**, and they can appear purely by chance in any sufficiently large dataset.

> Spurious correlation describes an association between two variables that do not actually share a meaningful relationship.

For example, the relationship between margarine consumption and Maine's divorce rate is entirely coincidental even with a correlation coefficient over 99%.
This illustrates the danger of blindly trusting data: despite the near-perfect statistical alignment, there is no logical or causal link between the two.

<center>
<img src="images/spurious_correlation.png" alt="Spurious correlation between margarine consumption and divorces." width="400"/>
</center>

Real-world data challenges are rarely as obvious as ice cream or margarine: they are nuanced and difficult to identify.
Even with domain expertise, analysts risk mistaking spurious correlations for genuine insights, often falling prey to confirmation bias.
The sheer scale of modern "big data" only increases the likelihood of mining these misleading, purely coincidental associations.


### 1.1.2: Challenge #2L Simpson's Paradox

Simpson's Paradox occurs when a trend observed in different groups of data disappears or reverses once those groups are combined.
It highlights the risk of drawing the wrong conclusion if you fail to account for a hidden "lurking" variable that influences the results.

> Simpson's Paradox occurs when a relationship between two variables disappears or reverses when taking another factor into account.

In 1973, UC Berkeley's admission data initially suggested a significant bias against women.
At the aggregate level, the overall admission rate for male applicants was substantially higher than for female applicants.
While the aggregate data shows a massive gap (30% for women versus 50% for men), the trend reverses when broken down by major.
In most individual departments, women actually had equal or higher admission rates than men.

<center>
<img src="images/simpson_on_berkley_1.png" alt="Possible Gender Bias in College Admissions." width="400"/><img src="images/simpson_on_berkley_2.png" alt="Admission Rate by Gender and Major." width="400"/>
</center>

The paradox arises because of how applicants were distributed: men tended to apply to departments with high acceptance rates, while women applied to more competitive ones with lower rates.
Even though women were often more successful within individual departments, the sheer volume of men in "easy" departments skewed the overall average.

<center>
<img src="images/simpson_on_berkley_3.png" alt="Admission Rate by Major and Percentage of Female Applicants." width="400"/>
</center>

When fitting a logistic regression model that includes both major and gender, the gender coefficient drops to near zero and actually slightly favours female applicants.
This confirms that once you account for the specific department, the apparent "male advantage" disappears.

```
Call:
glm(formula = admission_int ~ Gender + Major, data = admissions)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)
(Intercept)  0.739957   0.018508  39.980  < 2e-16 ***
GenderM     -0.007018   0.015084  -0.465    0.642
MajorB      -0.100760   0.021877  -4.606 4.21e-06 ***
MajorC      -0.387799   0.020768 -18.673  < 2e-16 ***
MajorD      -0.396616   0.020668 -19.190  < 2e-16 ***
MajorE      -0.485950   0.023599 -20.592  < 2e-16 ***
MajorF      -0.670465   0.021310 -31.462  < 2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for gaussian family taken to be 0.1903506)

    Null deviance: 1184.22  on 4838  degrees of freedom
Residual deviance:  919.77  on 4832  degrees of freedom
AIC: 5714.1

Number of Fisher Scoring iterations: 2
```

The Berkeley example shows that while Simpson's Paradox can flip a narrative, it doesn't automatically reveal the "truth": it just raises deeper questions about why those distributions exist.

In "the wild," it is rarely obvious which variables to include or exclude, and simply adding every available factor into a model can be as misleading as adding too few.
Traditional statistics offer little guidance on which result to trust, making it dangerous to base high-stakes decisions on a single coefficient without a causal framework.


### 1.1.3: Causality to the Rescue

Causal inference offers a systematic way to solve both spurious correlations and Simpson's Paradox.
While the next chapter covers the technical details, we'll start with a high-level look at how these methods work.

Causal inference is a formal framework for measuring causal effects, even from messy observational data.
It uses a rigorous scientific approach to outline assumptions and test whether a valid estimate is possible.
When these conditions are met, you can safely move beyond simple correlation to determine true cause and effect.

Causal inference requires explicitly defining relationships between variables; for instance, stating that $A$ _causes_ $B$ implies that $B$ _does not cause_ $A$.
By mapping these assumptions in a **Directed Acyclic Graph (DAG)**, we can identify which variables are independent and which must be controlled.
This formal structure provides a strong defence against Simpson's Paradox by ensuring we only adjust for variables when our causal logic requires it.

Defining causal relationships helps avoid the pitfalls of spurious correlations.
If your causal graph shows that two variables are **conditionally independent**, then any statistical association measured between them in your data is assumed to be **spurious**.
Consequently, that association is ignored and will not be allowed to bias your causal effect estimation.


## 1.2: A Shift in Perspective

Causal inference shifts the focus from predictive accuracy (ML) or significance testing (Statistics) to a structured scientific process.
This mental shift requires you to:

- **Formalise the question:** Define exactly what you want to know (e.g., _What if_ $X$ happened? _Why_ did $Y$ occur?)

- **Enumerate assumptions:** Map out how variables relate to one another in your specific environment.

- **Collect data:** Gather the necessary observations.

- **Verify identifiability:** Determine if a causal estimate is even possible based on your assumptions and the available data.

- **Estimate:** Calculate the actual causal effect.

In causal inference, calculating the **causal estimand** (the actual treatment effect) is often the easiest part: the maths rarely exceeds standard logistic regression or basic probability.

The real work happens upfront during **experimental design**.
You must:

- **Frame the question** precisely.

- **Consult domain experts** to validate your causal graph.

- **Verify identifiability**, determining if your data even allows for a causal answer, and if not, what new data is required.

A causal estimate is only as good as the assumptions behind it.
If your **causal model** (the DAG) accurately reflects reality, the resulting estimand is a true causal effect; if it doesn't, the model is wrong.

The power of this approach lies in its **falsifiability**.
By making your assumptions explicit in a graph, you allow others to challenge and refine them.
This iterative process, _refining the graph until the assumptions match the environment_, is what allows us to move beyond simple correlation.
It is both the primary challenge and the core strength of causal inference.


## 1.3: Causal Inference: A Brief History

To understand the technical details, it helps to see how these methods evolved.
This final section traces causality from its historical roots to its current application in modern industry.


### 1.3.1: Beginnings

While work dates back to the early 20th century, Donald Rubin's 1974 paper introduced the **Potential Outcome Framework**.

Rubin argued that while randomisation is ideal, using "carefully controlled" observational data is often a necessity.
He urged researchers to focus on variables (other than the treatment) that influence the outcome.
As seen with Simpson's Paradox, deciding which factors to control for is critical; Rubin's framework provided the first formal rigour for interpreting observational estimates as true causal effects.

> A "treatment" is any intervention, policy, or independent variable (such as a medical drug, a marketing skip-ad, or a price change) whose specific effect on an outcome you wish to measure.

Starting in the late 1980s, Judea Pearl transformed causal inference into a formal science, culminating in his 2000 textbook, _Causality_.
He pioneered the use of **Directed Acyclic Graphs (DAGs)** to map causal structures and developed **_do_-calculus**, a rigorous mathematical language for interventions.
This was groundbreaking because it provided the first formal rules for adjusting observational data to reach valid causal conclusions.

Pearl's **Ladder of Causation** ranks causal queries into three distinct _rungs_:

1. **Association (Prediction):** Seeing. How does one variable relate to another? (e.g., "Do admission rates differ by gender?")

2. **Intervention:** Doing. What happens if we force a change? (e.g., "What if we randomly assign students to different majors?")

3. **Counterfactuals:** Imagining. What would have happened in a different past? (e.g., "Would this specific student have been accepted if her gender had been different?")

Most standard data science sits on the first rung; causal inference is required to climb to the second and third.

<center>
<img src="images/causation_ladder.jpg" alt="Peral's Ladder of Causation." width="400"/>
</center>

Moving up the **Ladder of Causation** means asking increasingly causal questions.
The first rung covers **descriptive statistics** and standard modelling (like conditional expectations) which simply describe what occurred in the natural world.
These methods observe the data as it exists but cannot tell us what would happen if we intervened.

The second rung moves from observation to **action**.
These are interventional "what if" questions where a variable is fixed: either via a physical trial or mathematically through do-calculus.
While many predictive models try to answer "what if" by shifting input values, they are often just observing patterns rather than true interventions.
This rung seeks to understand how the system actually responds when you force a change.

The third rung involves **counterfactuals**: asking _why_ something happened or what _would_ have occurred in a hypothetical "dream world."
These questions are impossible to answer using interventional (2nd rung) tools alone.
Counterfactual reasoning allows us to pinpoint the causes of specific outcomes, calculate treatment effects for individuals rather than just averages, and explore scenarios that never actually happened.

Pearl and his contemporaries have since expanded these methods, which are now seeing widespread adoption in both academia and industry.

The following chapters will detail these techniques with practical examples and the necessary theory.
If you're interested in a deeper theoretical dive, Pearl's _Causality_ remains the definitive resource.


### 1.3.2: Initial Adoption and Applications

Economics and epidemiology have long sought to understand the effects of policy and behaviour, using quantitative methods well before Pearl's formalisation.

* **Economics:** Since the 1950s, economists used **Structural Equation Models (SEMs)**.
  Pearl argued these failed to distinguish between causal and statistical questions; his **Structural Causal Models (SCMs)** improved upon them by adding graphical modelling and handling non-linear relationships.

* **Epidemiology:** Classical methods successfully linked smoking to lung cancer but lacked a "formal basis" for testing hypotheses. Pearl's **_do_-calculus** provided this missing rigour.

While debates on adoption continue, both fields increasingly use Pearl's methods and the **potential outcomes framework** for policy evaluation and disease research.


### 1.3.3: Recent Applications in Industry

Causal inference has moved from theory to core industry practice, particularly where standard A/B testing falls short.

* **Personalised Marketing:** Companies use **uplift modelling** to target only the customers whose behaviour will actually change due to an ad (a counterfactual "what if").
  Uber developed the **CausalML** library specifically to scale this.

* **Product Features:** Netflix uses causal frameworks to understand how users respond to new features and to rank content in their recommendation engines.

* **Improving Experiments:** Traditional A/B tests can be slow, expensive, or impossible to run.
  Netflix uses double machine learning to augment these tests, matching similar observations to extract the "incremental" effect of a change without needing massive, long-term trials.


## 1.4: Books and Resources

### 1.4.1: Books

See the book for more details and pointers.

### 1.4.2: Tools

* DoWhy
* EconML
* CausalML
* CausalLift
* causal-learn
* Bayesian Model-Building Interface (Bambi)

### 1.4.3: Conferences and Workshops

* American Causal Inference Conference (ACIC)
* Causal Laerning and Reasoning (CLeaR)
* Miscellaneous: ICML, NeurIPS, ICLR


## 1.5: Conclusions and Looking Ahead

The key concepts that we have covered in this introductory chapter are the following:

* **Identifying Hidden Traps:** We saw how **Simpson's Paradox** and **spurious correlations** can lead to completely wrong conclusions if you only look at raw data.

* **The Scientific Solution:** We introduced **causal inference** as the formal rigour needed to navigate these traps and move beyond simple observation.

* **The Power of "What If":** We explored the **Ladder of Causation**, moving from basic prediction to interventional and counterfactual questions.

* **Real-World Impact:** We traced the history from Rubin and Pearl to modern applications like **Uber's uplift modelling** and **Netflix's experimental frameworks**.

Next, we'll dive into the actual mechanics: building the mathematical foundation and walking through the end-to-end process of solving a causal problem.
